# Satellite Remote-Sensing Screening Layer (Water Quality Monitoring System)

- **NDTI** (Normalized Difference Turbidity Index) — proxy for suspended sediment / turbidity
- **NDCI** (Normalized Difference Chlorophyll Index) — proxy for algal bloom / chlorophyll-a
- **TSS & CDOM Ratios**: Track heavy sediment loads and dissolved organic runoff.


## 1. ECC Setup

In [12]:
!pip install -q earthengine-api geemap pandas matplotlib

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

ee.Authenticate()
ee.Initialize(project='aqua-506015')

## 2. The Area of Interest (AOI)


In [13]:
aoi = ee.Geometry.Rectangle([80.34, 26.42, 80.40, 26.47])

Map = geemap.Map(center=[26.445, 80.37], zoom=12)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map

Map(center=[26.445, 80.37], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## 3. Load Sentinel-2 imagery and filter for usable (low-cloud) scenes

In [14]:
start_date = '2026-05-19'
end_date = '2026-08-19'

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

print('Number of usable scenes found:', s2.size().getInfo())

Number of usable scenes found: 12


## 4. Compute NDTI and NDCI for each scene
- $NDTI = \frac{Red - Green}{Red + Green}$ (Sentinel-2 Bands B4 & B3).

- $NDCI = \frac{RedEdge - Red}{RedEdge + Red}$ (Sentinel-2 Bands B5 & B4).

- $TSS = \frac{B4 (Red)}{B3 (Green)}$

- $CDOM = \frac{B2 (Blue)}{B3 (Green)}$

In [15]:
def mask_water(image):
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    water_mask = ndwi.gt(0)
    return image.updateMask(water_mask).addBands(ndwi)

s2_masked = s2.map(mask_water)

sample_water_pixels = s2_masked.first().select('NDWI').reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=10, maxPixels=1e9
).get('NDWI')
print('Water pixels found in AOI for first scene:', sample_water_pixels.getInfo())

def add_indices(image):
    ndti = image.normalizedDifference(['B4', 'B3']).rename('NDTI')
    ndci = image.normalizedDifference(['B5', 'B4']).rename('NDCI')

    tss_ratio = image.select('B4').divide(image.select('B3')).rename('TSS_ratio')

    cdom_ratio = image.select('B2').divide(image.select('B3')).rename('CDOM_ratio')
    return image.addBands([ndti, ndci, tss_ratio, cdom_ratio])

s2_indexed = s2_masked.map(add_indices)

Water pixels found in AOI for first scene: 331348


## 5. Build a time series (mean index value over the AOI, per scene)


In [16]:
def extract_values(image):
    stats = image.select(['NDTI', 'NDCI', 'TSS_ratio', 'CDOM_ratio']).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    )
    return ee.Feature(None, {
        'date': image.date().format('YYYY-MM-dd'),
        'NDTI': stats.get('NDTI'),
        'NDCI': stats.get('NDCI'),
        'TSS_ratio': stats.get('TSS_ratio'),
        'CDOM_ratio': stats.get('CDOM_ratio')
    })

ts_features = s2_indexed.map(extract_values).filter(ee.Filter.notNull(['NDTI', 'NDCI', 'TSS_ratio', 'CDOM_ratio']))
ts_data = ts_features.getInfo()['features']

df = pd.DataFrame([f['properties'] for f in ts_data])
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.head(10)

,CDOM_ratio,NDCI,NDTI,TSS_ratio,date
0,0.202432,0.135437,0.014705,1.034844,2026-05-19
1,0.837596,0.035943,0.006361,1.014017,2026-05-20
2,0.278813,0.224334,0.078255,1.246464,2026-05-22
3,0.948339,0.024131,-0.029682,0.943015,2026-05-25
4,0.196301,0.220792,0.016669,1.166748,2026-05-27
5,0.869891,0.065399,-0.064177,0.882868,2026-06-04
6,0.446482,0.132216,-0.115474,0.805371,2026-06-06
7,0.965242,-0.009451,-0.015139,0.974254,2026-06-09
8,0.924892,0.032625,-0.034646,0.935097,2026-06-14
9,0.677560,0.124133,-0.048338,0.922502,2026-06-16


## 6. Plot the time series

In [23]:
import plotly.express as px


df_melted = df.melt(id_vars=['date'],
                    value_vars=['NDTI', 'NDCI', 'TSS_ratio', 'CDOM_ratio'],
                    var_name='Index',
                    value_name='Value')


fig = px.line(df_melted, x='date', y='Value', color='Index', facet_row='Index',
              markers=True, title='Water Quality Indices Over Time (Interactive)')

fig.update_yaxes(matches=None)
fig.update_layout(height=800, hovermode='x unified')

fig.show()

## 7. Anomaly flagging

In [18]:
def flag_anomalies(df, column, n_std=2):
    mean_val = df[column].mean()
    std_val = df[column].std()
    threshold = mean_val + n_std * std_val
    df[f'{column}_anomaly'] = df[column] > threshold
    return df, threshold

df, ndti_threshold = flag_anomalies(df, 'NDTI')
df, ndci_threshold = flag_anomalies(df, 'NDCI')
df, tss_threshold = flag_anomalies(df, 'TSS_ratio')
df, cdom_threshold = flag_anomalies(df, 'CDOM_ratio')

print(f'NDTI anomaly threshold: {ndti_threshold:.4f}')
print(f'NDCI anomaly threshold: {ndci_threshold:.4f}')
print(f'TSS ratio anomaly threshold: {tss_threshold:.4f}')
print(f'CDOM ratio anomaly threshold: {cdom_threshold:.4f}')
print()

anomaly_cols = ['NDTI_anomaly', 'NDCI_anomaly', 'TSS_ratio_anomaly', 'CDOM_ratio_anomaly']
flagged = df[df[anomaly_cols].any(axis=1)]
print(f'{len(flagged)} of {len(df)} scenes flagged as anomalous:')
flagged[['date', 'NDTI', 'NDCI', 'TSS_ratio', 'CDOM_ratio'] + anomaly_cols]

NDTI anomaly threshold: 0.0809
NDCI anomaly threshold: 0.2487
TSS ratio anomaly threshold: 1.2334
CDOM ratio anomaly threshold: 1.2523

1 of 12 scenes flagged as anomalous:


,date,NDTI,NDCI,TSS_ratio,CDOM_ratio,NDTI_anomaly,NDCI_anomaly,TSS_ratio_anomaly,CDOM_ratio_anomaly
2,2026-05-22,0.078255,0.224334,1.246464,0.278813,False,False,True,False


## 8. Visualize a single flagged scene on the map


In [19]:
latest_image = s2_indexed.sort('system:time_start', False).first()

ndti_vis = {'min': -0.5, 'max': 0.5, 'palette': ['blue', 'white', 'brown']}
ndci_vis = {'min': -0.5, 'max': 0.5, 'palette': ['blue', 'white', 'green']}
tss_vis = {'min': 0.5, 'max': 2.0, 'palette': ['blue', 'white', 'sienna']}
cdom_vis = {'min': 0.5, 'max': 1.5, 'palette': ['darkslategray', 'white', 'blue']}

Map2 = geemap.Map(center=[26.445, 80.37], zoom=13)
Map2.addLayer(latest_image.clip(aoi).select('NDTI'), ndti_vis, 'NDTI (turbidity)')
Map2.addLayer(latest_image.clip(aoi).select('NDCI'), ndci_vis, 'NDCI (chlorophyll)')
Map2.addLayer(latest_image.clip(aoi).select('TSS_ratio'), tss_vis, 'TSS ratio (turbidity, secondary)')
Map2.addLayer(latest_image.clip(aoi).select('CDOM_ratio'), cdom_vis, 'CDOM ratio (organic matter)')
Map2.addLayer(aoi, {'color': 'red'}, 'AOI boundary', opacity=0.3)
Map2

Map(center=[26.445, 80.37], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…